In [ ]:
%pip install openai pydantic python-dotenv 

In [2]:
import os
import json
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

MODEL="llama-3.1-8b-instant"

In [3]:
# define structured schema for mail sorting
class EmailAnalysis(BaseModel):
    sender: str = Field(description="The name or entity sending the email.")
    core_issue: str = Field(description="A concise 1-sentence summary of what the user needs.")
    urgency_score: int = Field(description="Rate urgency from 1 (very low/spam) to 5 (critical/system down).")
    suggested_tone: str = Field(description="The emotional tone required for the response, e.g., empathetic, formal, apologetic.")

# simulate a local function that sends mail via SMTP
def send_automated_reply(recipient_email: str, email_body: str) -> str:
    """Simulates dispatching a response via an SMTP email server client layer."""
    print(f"✉️  [SMTP SERVER ACTIVATED]: Outgoing email dispatched successfully to <{recipient_email}>.")
    return "Status: DISPATCHED"

# define tool schema
email_tool = {
    "type": "function",
    "function": {  
        "name": "send_automated_reply",
        "description": "Send a generated email response to a specific recipient address.",
        "parameters": {
            "type": "object",
            "properties": {
                "recipient_email": {"type": "string", "description": "The destination email address."},
                "email_body": {"type": "string", "description": "The full text message body to transmit."}
            },
            "required": ["recipient_email", "email_body"],
            "additionalProperties": False
        }
    }
}
tools_list = [email_tool]

def main():
    print("--- 🤖 Starting Autonomous Email Triaging Agent ---")
    
    # mock incoming unstructured mails
    inbox_queue = [
        {
            "id": "msg_001",
            "from": "angry_client@enterprise.com",
            "body": "CRITICAL: Our production database just dropped connection and our API is throwing 500 errors! We are losing thousands of dollars every minute. Fix this immediately!"
        },
        {
            "id": "msg_002",
            "from": "marketing_bot@spamdeals.io",
            "body": "Hey friend! Want to save 20% on office chairs? Click this sketchy link right now to unlock your ultimate discount code before it expires!"
        }
    ]

    # process each email in the inbox queue
    for email in inbox_queue:
        print(f"\n================ STEP 1: TRIAGING INCOMING EMAIL ({email['id']}) ================")
        print(f"From: {email['from']}\nContent: {email['body']}")
        
        triage_prompt = f"Extract details from this email text:\n\"{email['body']}\""
        
        system_instruction = (
            "You are a rigid data extraction microservice. Analyze the user email and return "
            "exclusively a valid JSON object. You MUST use the exact property keys provided in "
            "this template, with no exceptions:\n\n"
            "{\n"
            '  "sender": "string value representing sender or organization",\n'
            '  "core_issue": "string containing a 1-sentence summary of the main issue",\n'
            '  "urgency_score": 5,\n' 
            '  "suggested_tone": "string describing the required response tone"\n'
            "}"
        )
        
        triage_response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": triage_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.1  # FIX: Kept at 0.1 for maximum precision without API rejections
        )
        
        raw_json_content = triage_response.choices[0].message.content
        
        try:
            analysis = EmailAnalysis.model_validate_json(raw_json_content)
            print(f"\n📊 [ANALYSIS RESULT]:")
            print(f" -> Sender:   {analysis.sender}")
            print(f" -> Issue:    {analysis.core_issue}")
            print(f" -> Urgency:  {analysis.urgency_score}/5")
            print(f" -> Tone Req: {analysis.suggested_tone}")
        except Exception as e:
            print(f"❌ Structural extraction parsing error: {e}")
            continue

        # choose temperature based on urgency score
        if analysis.urgency_score >= 4:
            current_temp = 0.1
            print("🚨 High Urgency Detected! Locking temperature to 0.1 for high precision.")
        elif analysis.urgency_score <= 1:
            print("🗑️  Email classified as low-priority or spam. Archiving message without action.")
            continue
        else:
            current_temp = 0.6
            print("Normal priority detected. Setting temperature to 0.6 for standard reply.")

        print(f"\n================ STEP 2: AUTONOMOUS EXECUTION LOOP ================")
        
        # build history context
        agent_memory = [
            {
                "role": "system", 
                "content": (
                    f"You are an executive customer relations auto-responder. Draft a message addressing the "
                    f"user's core issue: '{analysis.core_issue}'. Use a highly '{analysis.suggested_tone}' tone. "
                    f"You MUST call the send_automated_reply tool to finalize this task. "
                    f"Once the tool has been executed, explain to the user that it was sent successfully."
                )
            },
            {"role": "user", "content": f"Draft and send a reply to {email['from']} immediately."}
        ]
        
        while True:
            # generate request with the set temp
            response = client.chat.completions.create(
                model=MODEL,
                messages=agent_memory,
                tools=tools_list,
                temperature=current_temp
            )
            
            response_message = response.choices[0].message
            tool_calls = response_message.tool_calls
            
            if not tool_calls:
                print(f"\n✅ [Agent Loop Ended]: Final Tracking Summary:")
                print(response_message.content)
                break
                
            # append the agent response to memory for context
            agent_memory.append(response_message)
            
            for tool_call in tool_calls:
                if tool_call.function.name == "send_automated_reply":
                    args = json.loads(tool_call.function.arguments)
                    print(f"\n✍️  [AGENT DRAFTED REPLY]:\n{args.get('email_body')}\n")
                    
                    # execute local tool function
                    tool_status = send_automated_reply(
                        recipient_email=email['from'],
                        email_body=args.get('email_body')
                    )
                    
                    # append results back into memory
                    agent_memory.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": tool_status
                    })
            
            print(" -> Tool execution recorded. Transitioning to next turn naturally...")
            print("-" * 40)

    print("\n--- All inbox queue elements cleared successfully. ---")

if __name__ == "__main__":
    main()

--- 🤖 Starting Autonomous Email Triaging Agent ---

================ STEP 1: TRIAGING INCOMING EMAIL (msg_001) ================
From: angry_client@enterprise.com
Content: CRITICAL: Our production database just dropped connection and our API is throwing 500 errors! We are losing thousands of dollars every minute. Fix this immediately!

📊 [ANALYSIS RESULT]:
 -> Sender:   Our production team
 -> Issue:    Production database connection dropped, causing API 500 errors and significant financial loss.
 -> Urgency:  10/5
 -> Tone Req: Urgent and imperative
🚨 High Urgency Detected! Locking temperature to 0.1 for high precision.

================ STEP 2: AUTONOMOUS EXECUTION LOOP ================

✍️  [AGENT DRAFTED REPLY]:
Dear valued client, we are aware of the production database connection drop causing API 500 errors and significant financial loss. We are working on resolving this issue immediately. Please expect a follow-up email with a detailed update within the next 30 minutes. Your prom